# CoDHy v2.0 — Combinatorial Drug Hypothesis Generator


In [ ]:
!pip install --quiet neo4j python-dotenv pandas biopython thefuzz rdkit \
    sentence-transformers transformers torch accelerate bitsandbytes \
    krippendorff pingouin tabulate pykeen huggingface_hub spacy

# scispacy and its model pinned spacy<3.8, so skip their dependency resolution
!pip install --quiet scispacy --no-deps
!pip install --quiet scipy conllu pysbd joblib nmslib-metabrainz
!pip install --quiet --no-deps https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bionlp13cg_md-0.5.4.tar.gz

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scispacy 0.5.5 requires spacy<3.8.0,>=3.7.0, but you have spacy 3.8.18 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  P

In [ ]:
# Install and start Ollama locally in this Colab session (matches v1's proven setup)
!apt-get install -y zstd -q
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
ollama_process = subprocess.Popen("ollama serve", shell=True)
print("Starting Ollama server...")
time.sleep(5)

!ollama pull llama3.1
!ollama pull deepseek-r1:8b
!ollama pull qwen3:8b
!ollama pull medgemma1.5:4b
!ollama pull hf.co/mradermacher/JSL-MedLlama-3-8B-v9-GGUF:Q4_K_M
!ollama pull hf.co/MoMonir/Llama3-OpenBioLLM-8B-GGUF:Q4_K_M
!ollama pull hf.co/BioMistral/BioMistral-7B-GGUF:Q4_K_M

Reading package lists...
Building dependency tree...
Reading state information...
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Starting Ollama server...









In [ ]:
!ollama list

NAME                                                   ID              SIZE      MODIFIED           
hf.co/BioMistral/BioMistral-7B-GGUF:Q4_K_M             172ecaed0391    4.4 GB    About a minute ago    
hf.co/MoMonir/Llama3-OpenBioLLM-8B-GGUF:Q4_K_M         7d946486dbd0    4.9 GB    About a minute ago    
hf.co/mradermacher/JSL-MedLlama-3-8B-v9-GGUF:Q4_K_M    3e4ac813e599    4.9 GB    About a minute ago    
medgemma1.5:4b                                         433252621ab1    3.3 GB    About a minute ago    
qwen3:8b                                               500a1f067a9f    5.2 GB    About a minute ago    
deepseek-r1:8b                                         6995872bfe4c    5.2 GB    About a minute ago    
llama3.1:latest                                        46e0c10c039e    4.9 GB    About a minute ago    


In [ ]:
import importlib.util, pathlib, re
pkg = pathlib.Path(importlib.util.find_spec("en_ner_bionlp13cg_md").submodule_search_locations[0])
for cfg in pkg.rglob("config.cfg"):
    t = cfg.read_text()
    new = re.sub(r'(include_static_vectors\s*=\s*)"(True|False)"',
                 lambda m: m.group(1) + m.group(2).lower(), t)
    if new != t:
        cfg.write_text(new); print("patched", cfg)
import spacy
nlp = spacy.load("en_ner_bionlp13cg_md")

patched /usr/local/lib/python3.13/dist-packages/en_ner_bionlp13cg_md/en_ner_bionlp13cg_md-0.5.4/config.cfg


/usr/local/lib/python3.13/dist-packages/spacy/language.py:2235: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [ ]:
# import spacy, scispacy
# print(spacy.__version__)
# nlp = spacy.load("en_ner_bionlp13cg_md")
# doc = nlp("Gefitinib inhibits EGFR in lung cancer. EGFR is inhibited by gefitinib.")
# print([(e.text, e.label_) for e in doc.ents])
# print([(t.text, t.dep_, t.pos_) for t in doc][:12])

3.8.18
[('Gefitinib', 'SIMPLE_CHEMICAL'), ('EGFR', 'GENE_OR_GENE_PRODUCT'), ('lung cancer', 'CANCER'), ('EGFR', 'GENE_OR_GENE_PRODUCT'), ('gefitinib', 'SIMPLE_CHEMICAL')]
[('Gefitinib', 'nsubj', 'NOUN'), ('inhibits', 'ROOT', 'VERB'), ('EGFR', 'dobj', 'NOUN'), ('in', 'case', 'ADP'), ('lung', 'compound', 'NOUN'), ('cancer', 'nmod', 'NOUN'), ('.', 'punct', 'PUNCT'), ('EGFR', 'nsubjpass', 'NOUN'), ('is', 'auxpass', 'AUX'), ('inhibited', 'ROOT', 'VERB'), ('by', 'case', 'ADP'), ('gefitinib', 'nmod', 'NOUN')]


## 1  Imports

In [ ]:
import io, time, os, re, sys, ast, json, zipfile
from datetime import datetime
from collections import defaultdict

from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=True)
except Exception as e:
    print("Drive mount failed, attempting to force unmount stale connection...")
    !usr/bin/fusermount -u /content/drive
    drive.mount('/content/drive', force_remount=True)

PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/CoDHy"
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

import spacy
import scispacy
from pykeen.pipeline import pipeline
import requests
import urllib.parse
import pandas as pd
from rdkit import Chem
from Bio import Entrez
from thefuzz import process
from dotenv import load_dotenv
from neo4j import GraphDatabase
from scispacy.linking import EntityLinker
from sentence_transformers import SentenceTransformer, util

from build_kg import KnowledgeGraph
from nlp_processor import LiteratureProcessor
from fetch_data_util import (
    fetch_reactome_pathways, fetch_civic_database, fetch_gdc_data,
    fetch_clinical_trials, fetch_gene_metadata, fetch_chembl_data,
    fetch_string_interactions, fetch_synlethdb_local,
    get_depmap_direct_links, fetch_depmap_essentials,
    fetch_sider_data, fetch_drug_data_integrated,
)
from graph_embedding import GraphEmbeddingAgent
from hgenerator import DrugScreeningAgent
from hvalidator import ValidationAgent
from hranker import RankingAgent
from synergycheck import SynergyChecker
from metrics import calculate_ranking_metrics
from llm_backend import LLMBackend
from feedback_agent import FeedbackAgent
from taboo_memory import TabooMemory

pd.set_option('display.max_columns', None)
print("CoDHy v2.0 — imports OK")

Mounted at /content/drive


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
INFO:pykeen.utils:Using opt_einsum


CoDHy v2.0 — imports OK


## 2  Configuration


### 2.1  Genes & cancer projects of interest

In [ ]:
GENES_OF_INTEREST = [
    "MYC", "EGFR", "CCNE1", "MDM2", "PIK3CA", "PTEN", "TP53", "KRAS",
    "CHK1", "PARP1", "WEE1", "ATR", "AURKA", "CDK4", "CDK6", "ATM",
]

PROJECTS_OF_INTEREST = {
    "TCGA-GBM"  : {"full_name": "Glioblastoma Multiforme",                    "broad_disease": "Brain"},
    "TCGA-LGG"  : {"full_name": "Brain Lower Grade Glioma",                   "broad_disease": "Brain"},
    "TCGA-BRCA" : {"full_name": "Breast Invasive Carcinoma",                  "broad_disease": "Breast"},
    "TCGA-LUAD" : {"full_name": "Lung Adenocarcinoma",                        "broad_disease": "Lung"},
    "TCGA-LUSC" : {"full_name": "Lung Squamous Cell Carcinoma",               "broad_disease": "Lung"},
    "TCGA-COAD" : {"full_name": "Colon Adenocarcinoma",                       "broad_disease": "Colorectal"},
    "TCGA-READ" : {"full_name": "Rectum Adenocarcinoma",                      "broad_disease": "Colorectal"},
    "TCGA-SKCM" : {"full_name": "Skin Cutaneous Melanoma",                    "broad_disease": "Skin"},
    "TCGA-HNSC" : {"full_name": "Head and Neck Squamous Cell Carcinoma",      "broad_disease": "Head and Neck"},
    "TCGA-KIRC" : {"full_name": "Kidney Renal Clear Cell Carcinoma",          "broad_disease": "Kidney"},
    "TCGA-LIHC" : {"full_name": "Liver Hepatocellular Carcinoma",             "broad_disease": "Liver"},
    "TCGA-OV"   : {"full_name": "Ovarian Serous Cystadenocarcinoma",          "broad_disease": "Ovary"},
    "TCGA-PAAD" : {"full_name": "Pancreatic Adenocarcinoma",                  "broad_disease": "Pancreas"},
    "TCGA-PRAD" : {"full_name": "Prostate Adenocarcinoma",                    "broad_disease": "Prostate"},
    "TCGA-BLCA" : {"full_name": "Bladder Urothelial Carcinoma",               "broad_disease": "Bladder"},
    "TCGA-STAD" : {"full_name": "Stomach Adenocarcinoma",                     "broad_disease": "Stomach"},
}


### 2.2  Paths & directories

In [ ]:
# PROJECT_DIR   = "/content/drive/MyDrive/Colab Notebooks/CoDHy"
SAVE_PATH  = os.path.join(PROJECT_DIR, "data_cache/")
MODEL_PATH = os.path.join(PROJECT_DIR, "models/")
REPORT_DIR = os.path.join(PROJECT_DIR, "reports/")
OLLAMA_BASE_URL = "http://localhost:11434/api/generate"

for d in [SAVE_PATH, MODEL_PATH, REPORT_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"PROJECT_DIR  : {PROJECT_DIR}")
print(f"SAVE_PATH : {SAVE_PATH}")
print(f"MODEL_PATH: {MODEL_PATH}")
print(f"REPORT_DIR: {REPORT_DIR}")


PROJECT_DIR  : /content/drive/MyDrive/Colab Notebooks/CoDHy
SAVE_PATH : /content/drive/MyDrive/Colab Notebooks/CoDHy/data_cache/
MODEL_PATH: /content/drive/MyDrive/Colab Notebooks/CoDHy/models/
REPORT_DIR: /content/drive/MyDrive/Colab Notebooks/CoDHy/reports/


### 2.3  Secrets & credentials

In [ ]:
load_dotenv(os.path.join(PROJECT_DIR, "secrets.env"))

NEO4J_URI      = os.getenv("NEO4J_URI")
NEO4J_USER     = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
Entrez.email   = os.getenv("EMAIL", "you@example.com")

print("Neo4j URI :", NEO4J_URI)
print("Entrez e-mail :", Entrez.email)
print("Neo4j Username :", NEO4J_USER)
print("Neo4j Password :", NEO4J_PASSWORD)

Neo4j URI : neo4j+s://7ecec590.databases.neo4j.io
Entrez e-mail : basak.suvinava@mh-hannover.de
Neo4j Username : 7ecec590
Neo4j Password : qs9MFlSp4Qt6D8Ls4mCXt9_BwbKFoMxjIlVWRTh6r1I


## 3  Helper functions

### 3.1  Cached data loader

In [ ]:
def get_data(name, fetch_func, *args):
    """Load from CSV cache if present, otherwise fetch and cache."""
    fp = f"{SAVE_PATH}{name}.csv"
    if os.path.exists(fp):
        print(f"[cache] Loading {name}...")
        return pd.read_csv(fp).fillna('N/A')
    print(f"[fetch] {name}...")
    df = fetch_func(*args).fillna('N/A')
    df.to_csv(fp, index=False)
    return df


### 3.2  Report saver

In [ ]:
def save_report_json(results, report_dir=REPORT_DIR):
    """Persist final ranked results to a timestamped JSON file."""
    os.makedirs(report_dir, exist_ok=True)
    fn = os.path.join(report_dir, f"final_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json")
    with open(fn, 'w') as f:
        json.dump(results, f, indent=4)
    print(f"Report saved → {fn}")
    return fn

## 4  Knowledge Graph construction

Fetches data from 10 biomedical databases and loads them into Neo4j.  
Each source is cached locally so subsequent runs skip the network calls.


In [ ]:
def build_full_graph():
    try:
        print("Fetching data from databases...")
        df_civic         = get_data("civic",          fetch_civic_database,       GENES_OF_INTEREST)
        df_tcga_gdc      = get_data("tcga-gdc",       fetch_gdc_data,             GENES_OF_INTEREST, PROJECTS_OF_INTEREST)
        df_depmap_data   = get_data("depmap",         fetch_depmap_essentials,    GENES_OF_INTEREST)
        df_reactome      = get_data("reactome",       fetch_reactome_pathways,    GENES_OF_INTEREST)
        df_clinical_trial= get_data("clinical-trial", fetch_clinical_trials,      GENES_OF_INTEREST)
        df_chembl        = get_data("chembl",         fetch_chembl_data,          GENES_OF_INTEREST)
        df_string        = get_data("string",         fetch_string_interactions,  GENES_OF_INTEREST)
        df_synleth       = get_data("synleth",        fetch_synlethdb_local,      GENES_OF_INTEREST)
        df_gene_meta     = get_data("gene-meta",      fetch_gene_metadata,        GENES_OF_INTEREST)
        df_drugs         = get_data("drugs",          fetch_drug_data_integrated, GENES_OF_INTEREST)

        kg.wipe_database()
        kg.create_constraints()

        print("Loading data into the graph...")
        kg.load_civic_data(df_civic)
        kg.load_gdc_data(df_tcga_gdc)
        kg.load_depmap_data(df_depmap_data)
        kg.load_reactome_pathways(df_reactome)
        kg.load_clinical_trials(df_clinical_trial)
        kg.load_chembl_data(df_chembl)
        kg.load_string_interactions(df_string)
        kg.load_synleth_smart_mapping(df_synleth)
        kg.load_gene_metadata(df_gene_meta)
        kg.load_drug_metadata_consensus(df_drugs)
        kg.enrich_drugs_with_sider()

        print("KG build complete.")
    except Exception as e:
        print(f"KG build error: {e}")
        raise


## 5  Hypothesis generation, validation & ranking

**Methods:**

| `method` | What runs |
|---|---|
| `"Full"` | Literature mining + graph embeddings + LLM |
| `"No-Node2Vec"` | Literature mining + LLM (no embeddings) |
| `"LLM-only"` | LLM alone, no KG context |

When `method="Full"`, `embedding_model` selects which pykeen KGE model
learns the Layer-2/Layer-3 graph embeddings: `"RotatE"` (default) or
`"TransE"`. This is an architecture ablation, independent of the
`method` dimension above.

**Pipeline (runs once per experiment):**

```
generate v1 -> validate v1 -> FeedbackAgent(v1) -> update taboo memory
-> generate v2 (revised, feedback+taboo aware) -> validate v2 -> rank v2
```

- **FeedbackAgent** answers "how can this hypothesis be improved?" — distinct
  from ValidationAgent, which answers "is this hypothesis supported by
  evidence?". Its suggestions feed into the round-2 generation prompt.
- **TabooMemory** persists rejected combinations (low safety score, or weak
  evidence per FeedbackAgent's rejection check) to `taboo_memory.json`
  across runs, and blocks the generator from reproposing them.
- **Chain-of-thought**: each hypothesis now carries a `reasoning` field
  (the LLM's step-by-step justification), kept distinct from the concise
  `mechanism` field, and reported in the final output for qualitative
  analysis.


In [ ]:
def run_full_experiment(kg, INPUT_GENE, INPUT_DISEASE, method="Full", num_lit=50,
                         model="llama3.1", hypo_count=4, embedding_model="RotatE",
                         email=Entrez.email, ollama_base_url=OLLAMA_BASE_URL,
                         taboo_path=os.path.join(SAVE_PATH, "taboo_memory.json")):

    _t0 = time.perf_counter()
    generator      = DrugScreeningAgent(kg, model_name=model, ollama_base_url=ollama_base_url)
    validator      = ValidationAgent(kg, model_name=model, ollama_base_url=ollama_base_url, email=email)
    feedback_agent = FeedbackAgent(kg, model_name=model, ollama_base_url=ollama_base_url)
    ranker         = RankingAgent(kg)
    taboo          = TabooMemory(storage_path=taboo_path)

    try:
        if method in ["Full", "No-Embed"]:
            print("Fetching dynamic ontology from Neo4j...")
            CURRENT_NODES, CURRENT_RELATIONS = kg.get_ontology_context(
                target_labels=['Gene', 'Drug', 'Disease'])
            nlp = LiteratureProcessor(existing_nodes=CURRENT_NODES,
                                       existing_relations=CURRENT_RELATIONS, email=email)
            papers = nlp.fetch_papers(INPUT_GENE, INPUT_DISEASE, max_results=num_lit)
            if not papers:
                print(f"No papers found in PubMed for given {INPUT_GENE} and {INPUT_DISEASE}...")
            triples = nlp.extract_triples(papers)
            n_dep = sum(1 for t in triples if t.get("extraction_method") == "dependency-parsed")
            print(f"Literature Mining Complete: {len(papers)} abstracts, {len(triples)} triples "
                  f"({n_dep} dependency-parsed, {len(triples)-n_dep} co-occurrence-fallback).")
            kg.load_literature_triples(triples)

        embedder = None
        if method == "Full":
            embedder = GraphEmbeddingAgent(kg, model_name=embedding_model)
            EMBEDDING_FILE = os.path.join(MODEL_PATH, f"graph_embeddings_{embedding_model.lower()}")
            if os.path.exists(EMBEDDING_FILE + ".pt"):
                print(f"Loading existing {embedding_model} model...")
                embedder.load_embeddings(EMBEDDING_FILE)
            else:
                embedder.generate_embeddings(file_path=EMBEDDING_FILE)

        print("\n--- [Round 1] Generating Hypotheses ---")
        taboo_ctx_v1 = taboo.get_taboo_list(INPUT_GENE, INPUT_DISEASE)
        if taboo_ctx_v1:
            print(f"  ({len(taboo_ctx_v1)} previously-rejected combinations loaded from taboo memory)")
        hypotheses_v1 = generator.generate_hypothesis(
            INPUT_DISEASE, INPUT_GENE, embedding_agent=embedder, method=method,
            hypo_count=hypo_count, taboo_context=taboo_ctx_v1,
        )
        if not hypotheses_v1:
            print("No hypotheses generated from round 1.")
            return None

        print("\n--- [Round 1] Validating Hypotheses ---")
        validated_v1 = validator.validate_batch(hypotheses_v1)

        print("\n--- Feedback Agent: reviewing round 1 ---")
        feedback_list = feedback_agent.generate_feedback(validated_v1)
        feedback_agent.print_feedback_report(feedback_list)

        for h, fb in zip(validated_v1, feedback_list):
            if fb["is_rejected"]:
                taboo.add_rejected(
                    INPUT_GENE, INPUT_DISEASE, h.get("combination", ""),
                    reason=fb["rejection_reason"], score=h.get("safety_score"),
                )
        print(f"\nTaboo memory now holds {taboo.count(INPUT_GENE, INPUT_DISEASE)} "
              f"rejected combination(s) for {INPUT_GENE} / {INPUT_DISEASE}.")

        print("\n--- [Round 2] Generating Revised Hypotheses ---")
        taboo_ctx_v2 = taboo.get_taboo_list(INPUT_GENE, INPUT_DISEASE)
        hypotheses_v2 = generator.generate_hypothesis(
            INPUT_DISEASE, INPUT_GENE, embedding_agent=embedder, method=method,
            hypo_count=hypo_count, taboo_context=taboo_ctx_v2, feedback_context=feedback_list,
        )
        if not hypotheses_v2:
            print("No hypotheses generated (round 2) — falling back to round 1 results.")
            validated_final = validated_v1
        else:
            print("\n--- [Round 2] Validating Revised Hypotheses ---")
            validated_final = validator.validate_batch(hypotheses_v2)

        print("\n--- Ranking Hypotheses ---")
        final_ranked_list = ranker.rank(validated_final)

        return {
            "ranked_hypotheses": final_ranked_list,
            "round1_feedback": feedback_list,
            "taboo_context_used": taboo_ctx_v2,
            "runtime_seconds": time.perf_counter() - _t0,
        }

    except Exception as e:
        print(f"Error Occured in hypotheses handling phase: {e}")
        return None

## 6  Initialise Knowledge Graph

Connects to Neo4j and builds the graph from scratch if it is empty.


In [ ]:
kg = KnowledgeGraph(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)

with kg.driver.session() as session:
    node_count = session.run("MATCH (n) RETURN count(n) as c").single()["c"]

if node_count == 0:
    print("Graph is empty. Building from scratch...")
    build_full_graph()
else:
    print(f"Graph already has {node_count} nodes — skipping build.")


Successfully connected to Neo4j.
Graph already has 19566 nodes — skipping build.


## 7  Experiment parameters

Edit the variables below to target a different gene / cancer type.


In [ ]:
INPUT_GENE       = "EGFR"
INPUT_DISEASE    = "Glioblastoma Multiforme"
CHOSEN_METHOD    = "Full"       # "Full" | "No-Embed" | "LLM-only"
EMBEDDING_MODEL  = "RotatE"     # "RotatE" | "TransE" (only used when CHOSEN_METHOD == "Full")
LIT_TO_FETCH     = 50
# MODEL            = "meta-llama/Llama-3.1-8B-Instruct" # For HuggineFace Inference Client
MODEL            = "llama3.1"                         # For Ollama
HYPO_COUNT       = 3

## 8  Run experiment

In [ ]:
report = run_full_experiment(
    kg, INPUT_GENE, INPUT_DISEASE,
    method=CHOSEN_METHOD, num_lit=LIT_TO_FETCH,
    model=MODEL, hypo_count=HYPO_COUNT,
    embedding_model=EMBEDDING_MODEL,
)

Fetching dynamic ontology from Neo4j...
Initializing NLP Pipeline...


/usr/local/lib/python3.13/dist-packages/spacy/util.py:971: UserWarning: [W095] Model 'en_ner_bionlp13cg_md' (0.5.4) was trained with spaCy v3.7.4 and may not be 100% compatible with the current version (3.8.18). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


Loaded BioNLP model for NER.
Loading Sentence Transformer...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Articles fetched: 50
Results extracted: 50
Literature Mining Complete: 50 abstracts, 1327 triples (1262 dependency-parsed, 65 co-occurrence-fallback).
Loading 1327 NLP-extracted triples (EXPERIMENTAL) into the knowledge graph...


INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()


✅ Loaded 1327 triples into the graph.
Loading existing RotatE model...
✅ RotatE model loaded from /content/drive/MyDrive/Colab Notebooks/CoDHy/models/graph_embeddings_rotate.pt.

--- [Round 1] Generating Hypotheses ---

Agent is thinking about EGFR in Glioblastoma Multiforme using Full method...
   VANILLA check: 5000/5000 paths passed, 0 flagged for inconsistency.
      Successfully generated 4 hypotheses.

--- [Round 1] Validating Hypotheses ---

Validation Agent is reviewing candidates...
   -> Reviewing Hypothesis 1: Selumetinib + UPMD1
   -> Reviewing Hypothesis 2: Selumetinib + Ca9-22
   -> Reviewing Hypothesis 3: Selumetinib + SNU-2292
   -> Reviewing Hypothesis 4: Selumetinib + NCI-H28

--- Feedback Agent: reviewing round 1 ---

  FEEDBACK AGENT REPORT

[1] Selumetinib + UPMD1  —  kept
    Improvement suggestion: To strengthen this hypothesis, it would be beneficial to identify a specific molecular interaction between EGFR and the target of UPMD1, and propose a mechanistic step

## 9  Ranking metrics

In [ ]:
if report:
    results = report["ranked_hypotheses"]
    mrr_score, ndcg_score = calculate_ranking_metrics(results, k=3)

    print("=" * 80)
    print("CoDHy FINAL REPORT")
    print("=" * 80)
    print(f"Cancer Type: {INPUT_DISEASE}")
    print(f"Gene in context: {INPUT_GENE}")
    print(f"Generation Method: {CHOSEN_METHOD}")
    print("-" * 50)
    print(f"MRR (First Valid Hit): {mrr_score:.4f}")
    print(f"nDCG@3 (Ranking Quality): {ndcg_score:.4f}")
    print("-" * 50)

    synergy = SynergyChecker(kg, SAVE_PATH)

    for i, h in enumerate(results, 1):
        combo_str = h.get('combination', '')
        drugs = [d.strip() for d in combo_str.split("+") if d.strip()]
        drug1, drug2 = drugs[:2]
        synergy_summary = synergy.check_drugcomb_synergy(drug1, drug2)
        h['synergy'] = synergy_summary if synergy_summary else "No direct data in DrugComb database for this combination."

        print(f"\nRANK {i}: {combo_str}")
        print(f"   Rationale: {h.get('rationale_type', 'N/A')}")
        if h.get('reasoning'):
            print(f"   Reasoning (CoT): {h.get('reasoning')}")
        print(f"   Mechanism: {h.get('mechanism')}")
        print(f"   Verdict: {h.get('verdict', 'N/A')}")

        print(f"   SCORES:")
        print(f"     - Composite: {h.get('ranking_score')}/10")
        print(f"     - Safety (AI): {h.get('safety_score')}/10")
        print(f"     - Evidence (Graph): {h.get('graph_evidence_score'):.1f}/10.0")
        print(f"     - Biological plausibility: {h.get('biological_plausibility_score', 'N/A')}")
        print(f"     - Literature support: {h.get('literature_support_score', 'N/A')}")
        print(f"     - Toxicity risk: {h.get('toxicity_risk_score', 'N/A')}")
        print(f"     - Mechanistic coherence: {h.get('mechanistic_coherence_score', 'N/A')}")

        print(f"   CRITIQUE:")
        print(f"     - Novelty-Exact: {h.get('is_novel_exact', 'N/A')} with {h.get('ne_hits', 0)} hits")
        print(f"     - Novelty-in-Context: {h.get('is_novel_context', 'N/A')} with {h.get('nc_hits', 0)} hits")
        print(f"     - Synergy: {h['synergy']}")
        print(f"     - Remarks: {h.get('critique')}")
        print(f"     - Supporting evidence: {h.get('supporting_evidence')}")

        print(f"   EVIDENCE SOURCES:")
        all_sources = list(set(h.get('source_urls', [])))
        pubmed_links = [url for url in all_sources if "pubmed.ncbi.nlm.nih.gov" in url]

        print(f"     [Clinical & Knowledge Databases]:")
        for db_key in ['civic_urls', 'chembl_urls', 'trial_urls', 'reactome_urls']:
            urls = h.get(db_key, [])
            if urls:
                for url in urls[:3]:
                    print(f"       {url}")
            else:
                print(f"       No direct records found.")

        print(f"     [Research Literature - PubMed]:")
        if pubmed_links:
            for url in pubmed_links[:4]:
                print(f"       {url}")
        else:
            print(f"       No specific combination literature found.")

#     print("\n" + "=" * 80)
#     print("ROUND 1 FEEDBACK (informed the revision above)")
#     print("=" * 80)
#     for fb in report["round1_feedback"]:
#         status = "REJECTED -> taboo list" if fb["is_rejected"] else "kept"
#         print(f"\n[{fb['id']}] {fb['combination']} — {status}")
#         if fb["is_rejected"]:
#             print(f"    Reason: {fb['rejection_reason']}")
#         print(f"    Suggestion: {fb['improvement_suggestions']}")
# else:
#     print("No results to report.")

CoDHy FINAL REPORT
Cancer Type: Glioblastoma Multiforme
Gene in context: EGFR
Generation Method: Full
--------------------------------------------------
MRR (First Valid Hit): 0.0000
nDCG@3 (Ranking Quality): 0.0000
--------------------------------------------------


DriverError: Driver closed

## 10  Save report & close

In [ ]:
if report:
    report_path = save_report_json(report)

kg.close()
print("Neo4j connection closed.")

Report saved → /content/drive/MyDrive/Colab Notebooks/CoDHy/reports/final_report_20261113_112610.json
Neo4j connection closed.


## 11  Run Evaluation as LLM-as-a-Judge

In [ ]:
# import llm_judge, judge_metrics, baseline_generators, objective_metrics

# if report:
#     codhy_hyps = report["ranked_hypotheses"]

#     # --- 1. Zero-shot baselines: same gene/disease/hypo_count as CoDHy ---
#     _t0 = time.perf_counter()
#     baselines = baseline_generators.generate_all_baselines(
#         INPUT_DISEASE, INPUT_GENE, hypo_count=HYPO_COUNT,
#         hf_token=os.environ.get("HF_TOKEN"),
#     )
#     baseline_runtimes = {name: (time.perf_counter() - _t0) / max(len(baselines), 1) for name in baselines}

#     # --- 2. Individual-score judging (blind, self-exclusion applied), 3 repeats ---
#     all_for_judging = codhy_hyps + [h for hyps in baselines.values() for h in hyps]
#     judge_df = llm_judge.run_judge_panel(
#         all_for_judging, only_method=None, n_repeats=1,
#         hf_token=os.environ.get("HF_TOKEN"),
#     )
#     judge_df.to_csv(os.path.join(SAVE_PATH, "llm_judge_scores.csv"), index=False)

#     # --- 3. Pairwise CoDHy vs. each baseline (paper's Table 3, tab:pairwise-results) ---
#     pairwise_frames = []
#     for system_name, hyps in baselines.items():
#         pw = llm_judge.run_pairwise_panel(
#             codhy_hyps, hyps, "CoDHy-Full", system_name,
#             hf_token=os.environ.get("HF_TOKEN"),
#         )
#         pairwise_frames.append(pw)
#     pairwise_df = pd.concat(pairwise_frames, ignore_index=True) if pairwise_frames else pd.DataFrame()
#     pairwise_df.to_csv(os.path.join(SAVE_PATH, "pairwise_comparisons.csv"), index=False)

#     table3_pairwise = judge_metrics.pairwise_report(pairwise_df)
#     print("\n--- Table 3 (tab:pairwise-results): pairwise win/tie/loss ---")
#     print(table3_pairwise.to_markdown(index=False))
#     table3_pairwise.to_csv(os.path.join(SAVE_PATH, "table3_pairwise_results.csv"), index=False)

#     # --- 4. Full judge report: Table 1 (tab:judge-results), Table 2 (tab:judge-diagnostics),
#     #     plus supplementary diagnostics. build_full_report() already prints every one of
#     #     these to console automatically -- nothing extra needed for the printing part.
#     eval_report = judge_metrics.build_full_report(judge_df, codhy_hyps)

#     # Save EVERY table build_full_report produced to its own CSV, matching the paper's
#     # own claim that "full per-judge scores, pairwise comparisons, agreement statistics...
#     # are provided in the repository."
#     for name, df in eval_report.items():
#         out_path = os.path.join(SAVE_PATH, f"eval_{name}.csv")
#         df.to_csv(out_path, index=(name == "judge_bias"))

#     # --- 5. Objective metrics table (Evidence Coverage / Novel Combination Rate /
#     #     Diversity / Citation Precision / Runtime) -- from the supervisor's original
#     #     evaluation plan. NOT one of the 3 tables in the LaTeX you pasted last turn
#     #     (judge-results, judge-diagnostics, pairwise-results) -- check your current full
#     #     paper draft on whether this is still wanted; comment this block out if not.
#     system_to_hyps = {"CoDHy-Full": codhy_hyps, **baselines}
#     runtimes = {"CoDHy-Full": report.get("runtime_seconds"), **baseline_runtimes}
#     objective_metrics_table = objective_metrics.build_objective_metrics_table(
#         system_to_hyps, runtime_seconds=runtimes
#     )
#     print("\n--- Objective metrics: Evidence Coverage / Novel Combination Rate / Diversity / Citation Precision / Runtime ---")
#     print(objective_metrics_table.to_markdown(index=False))
#     objective_metrics_table.to_csv(os.path.join(SAVE_PATH, "objective_metrics.csv"), index=False)

# else:
#     print("Cannot run LLM judge: No report generated from the experiment.")